# Imports

In [1]:
import os
import scvelo as scv
import scanpy as sc
import numpy as np
import plotly.express as px
import pandas as pd
import scipy.sparse as sp
import einops
import umap
import gseapy as gp
from gseapy import enrichr
import torch
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from scbmlp.utils import calculate_freqs
from scbmlp.datasets import get_regression_datasets
from scbmlp.models import ScBMLPRegressor, Config

In [2]:
# Set random seeds for reproducibility
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)  # For CUDA if available
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Set scanpy settings for deterministic behavior
sc.settings.verbosity = 2  # Reduce verbosity
sc.settings.set_figure_params(dpi=80, facecolor='white')

In [3]:
# Create folder to store figures
os.makedirs("figures/frequency/", exist_ok=True)

# Set figure params for blog dimensions
fig_width = 800
title_fontsize = 18
legend_fontsize = 14

# Load data

In [4]:
n_nbrs = 10
n_freqs = 4
n_genes = 10_000
layer = "spliced"  # same as adata.X
device = "cpu"
random_state = 42

adata = scv.datasets.pancreas()
adata = adata[:, ~adata.var_names.str.startswith(("Rpl", "Rps"))]  # remove ribosomal genes
sc.pp.normalize_total(adata, layer=layer)
sc.pp.log1p(adata, layer=layer)
sc.pp.highly_variable_genes(adata, subset=True, n_top_genes=n_genes, layer=layer)

calculate_freqs(
    adata,
    n_nbrs=n_nbrs,
    n_freqs=n_freqs,
    layer=layer,
    sparse_decomp=False,
    device=device,
)

train_dataset, val_dataset, _ = get_regression_datasets(
    adata,
    target_key="X_freq",
    train_split=0.7,
    val_split=0.3,
    layer=layer,
    random_state=random_state,
    device=device,
)

normalizing counts per cell
    finished (0:00:01)
extracting highly variable genes
    finished (0:00:00)


### Visualize

In [5]:
cmap = "RdBu_r"

In [6]:
for i in range(4):
    fig = px.scatter(
        x=adata.obsm["X_umap"][:, 0],
        y=adata.obsm["X_umap"][:, 1],
        color=adata.obsm["X_freq"][:, i],
        color_continuous_scale=cmap,
        color_continuous_midpoint=0,
        title=f"Frequency {i} (λ={adata.uns['freq_vals'][i]:.4f})",
        labels={"x": "UMAP1", "y": "UMAP2"},
        hover_data=[adata.obs["clusters"]],
        width=fig_width,
    )
    fig.update_traces(marker=dict(size=3))

    # Apply custom font sizes from earlier variables
    fig.update_layout(
        title_font=dict(size=title_fontsize),
        font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
        legend=dict(font=dict(size=legend_fontsize)),
        margin=dict(l=250, r=250, t=100, b=50),
        coloraxis_colorbar=dict(
            title="frequency", 
            title_font=dict(size=legend_fontsize),
            title_side="right"  # Position title on the right side of the colorbar
        )
    )
    # Axis title font tuning and remove ticks (UMAP coordinates are not meaningful)
    axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
    fig.update_xaxes(title_font=dict(size=axis_title_size), showticklabels=False, ticks="")
    fig.update_yaxes(title_font=dict(size=axis_title_size), showticklabels=False, ticks="")

    fig.show()

    # Save combined figure
    # fig.write_html(f"figures/frequency/frequency_{i}_umap.html")
    fig.write_image(f"figures/frequency/frequency_{i}_umap.png", width=fig_width, height=400, scale=2)

In [7]:
# Make cell type colormap
unique_clusters = adata.obs["clusters"].unique()

# colors = px.colors.qualitative.Set1[:len(unique_clusters)]  # High contrast, scientific
# colors = px.colors.qualitative.Dark2[:len(unique_clusters)]  # Darker colors
# colors = px.colors.qualitative.T10[:len(unique_clusters)]    # Tableau colors
# colors = px.colors.qualitative.Pastel1[:len(unique_clusters)]  # Softer colors
colors = px.colors.qualitative.Plotly[:len(unique_clusters)]  # Original Plotly colors
# colors = sc.pl.palettes.default_28[:len(unique_clusters)]     # Scanpy default

# import plotly.colors as pc
# colors = pc.sample_colorscale(cmap, [i/(len(unique_clusters)-1) for i in range(len(unique_clusters))])

cluster_colors = {cluster: colors[i] for i, cluster in enumerate(unique_clusters)}

# Train

In [8]:
d_hidden = 128
n_epochs = 100
lr = 1e-4
device = "cpu"
batch_size = 64  # no more (poor loss), no less (slower)

In [9]:
cfg = Config(
    d_input=n_genes,
    d_hidden=d_hidden,
    d_output=n_freqs,
    n_epochs=n_epochs,
    lr=lr,
    device=device,
    batch_size=batch_size,
    bias=True,
    seed=random_state
)

model = ScBMLPRegressor(cfg, loss_fn="huber")
train_losses, train_metrics, val_losses, val_metrics = model.fit(
    train_dataset, val_dataset,
)

Training for 100 epochs: 100%|██████████| 100/100 [01:11<00:00,  1.41it/s, train_loss=0.0000, train_mae=0.0009, val_loss=0.0000, val_mae=0.0035]


In [10]:
# Create combined plot with loss and MAE subplots

# Prepare data
loss_df = pd.DataFrame({
    'Epoch': list(range(len(train_losses))) + list(range(len(val_losses))),
    'Loss': train_losses + val_losses,
    'Type': ['Train'] * len(train_losses) + ['Validation'] * len(val_losses)
})

metric_df = pd.DataFrame({
    'Epoch': list(range(len(train_metrics))) + list(range(len(val_metrics))),
    'MAE': train_metrics + val_metrics,
    'Type': ['Train'] * len(train_metrics) + ['Validation'] * len(val_metrics)
})

# Create subplots
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Training and Validation Loss', 'Training and Validation MAE'],
    vertical_spacing=0.15
)

# Colors shared across metrics
colors = {'Train': 'blue', 'Validation': 'red'}

# Add loss plot traces (these will own the legend entries)
for type_name in ['Train', 'Validation']:
    type_data = loss_df[loss_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['Loss'],
            mode='lines',
            name=type_name,               # Single legend entry per type
            legendgroup=type_name,
            showlegend=True,              # Only show legend for loss traces
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} Loss: %{{y:.4f}}<extra></extra>"
        ),
        row=1, col=1
    )

# Add MAE plot traces (no new legend entries, grouped with loss) - now solid lines
for type_name in ['Train', 'Validation']:
    type_data = metric_df[metric_df['Type'] == type_name]
    fig.add_trace(
        go.Scatter(
            x=type_data['Epoch'],
            y=type_data['MAE'],
            mode='lines',
            name=type_name,               # Same name but suppressed in legend
            legendgroup=type_name,
            showlegend=False,
            line=dict(color=colors[type_name], width=2),
            hovertemplate=f"Epoch %{{x}}<br>{type_name} MAE: %{{y:.4f}}<extra></extra>"
        ),
        row=2, col=1
    )

# Update layout (dimensions & base styling)
fig.update_layout(
    height=500,
    width=fig_width,
    title_text="Training Progress: Loss and MAE",
    showlegend=True,
    legend=dict(title=None, orientation='h', yanchor='bottom', y=1.02, xanchor='left', x=0),
    margin=dict(l=80, r=80, t=80, b=60)
)

# Axis labels
fig.update_xaxes(title_text="Epoch", row=2, col=1)
fig.update_yaxes(title_text="Loss", row=1, col=1)
fig.update_yaxes(title_text="MAE", row=2, col=1)

# Apply custom font sizes
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    legend=dict(font=dict(size=legend_fontsize))
)

# Axis title and tick font tuning
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))
fig.update_xaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))
fig.update_yaxes(title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size))

fig.show()

# Save plot
# fig.write_html("figures/frequency/frequency_training.html")
fig.write_image("figures/frequency/frequency_training.png", width=fig_width, height=600, scale=2)

In [11]:
# Get predictions on validation set to analyze error distribution
model.eval()
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

all_predictions = []
all_targets = []

with torch.no_grad():
    for batch in val_loader:
        inputs, targets = batch
        predictions = model(inputs)
        all_predictions.append(predictions.cpu())
        all_targets.append(targets.cpu())

# Concatenate all batches
predictions = torch.cat(all_predictions, dim=0)
targets = torch.cat(all_targets, dim=0)

# Calculate absolute errors per frequency component
absolute_errors = torch.abs(predictions - targets)  # Shape: (n_samples, n_freqs)

In [12]:
# Calculate per-frequency MAE (mean absolute error across all cells)
per_freq_mae = absolute_errors.mean(dim=0).numpy()

# Calculate per-cell MAE (mean absolute error across all frequencies)
per_cell_mae = absolute_errors.mean(dim=1).numpy()

# Create subplots with increased horizontal spacing
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Per-Frequency MAE Distribution',
        'Per-Cell MAE Distribution', 
        'Absolute Errors by Frequency Component',
        'Frequency Component Error Analysis'
    ],
    specs=[[{"type": "xy"}, {"type": "xy"}],
           [{"type": "xy"}, {"type": "xy"}]],
    horizontal_spacing=0.15,  # Increased from default to prevent column collision
    vertical_spacing=0.12
)

# 1. Bar plot of per-frequency MAE (since we only have 5 frequencies)
freq_names = [f"Freq {i}" for i in range(n_freqs)]
fig.add_trace(
    go.Bar(x=freq_names, y=per_freq_mae, name="Per-Frequency MAE"),
    row=1, col=1
)

# 2. Histogram of per-cell MAE
fig.add_trace(
    go.Histogram(x=per_cell_mae, nbinsx=50, name="Per-Cell MAE"),
    row=1, col=2
)

# 3. Box plot of errors for each frequency component
for freq_idx in range(n_freqs):
    fig.add_trace(
        go.Box(y=absolute_errors[:, freq_idx].numpy(), name=f"Freq {freq_idx}", showlegend=False),
        row=2, col=1
    )

# 4. Scatter plot showing relationship between frequency eigenvalues and prediction errors
freq_eigenvals = adata.uns['freq_vals'][:n_freqs]
fig.add_trace(
    go.Scatter(
        x=freq_eigenvals, 
        y=per_freq_mae, 
        mode='markers+text',
        text=freq_names,
        textposition="top center",
        marker=dict(size=10),
        name="Eigenvalue vs MAE"
    ),
    row=2, col=2
)

fig.update_layout(
    height=800,
    width=fig_width,
    title_text="Transcriptional Frequency Prediction Error Analysis",
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    showlegend=False
)

# Update axis labels with proper font sizing
axis_title_size = max(legend_fontsize - 2, int(0.85 * legend_fontsize))
axis_tick_size = max(legend_fontsize - 4, int(0.7 * legend_fontsize))

fig.update_xaxes(title_text="Frequency Component", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=1)
fig.update_yaxes(title_text="MAE", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=1)
fig.update_xaxes(title_text="MAE", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=2)
fig.update_yaxes(title_text="Count", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=1, col=2)
fig.update_xaxes(title_text="Frequency Components", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=2, col=1)
fig.update_yaxes(title_text="Absolute Error", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=2, col=1)
fig.update_xaxes(title_text="Frequency Eigenvalue", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=2, col=2)
fig.update_yaxes(title_text="MAE", title_font=dict(size=axis_title_size), tickfont=dict(size=axis_tick_size), row=2, col=2)

fig.show()

# Save plot
# fig.write_html("figures/frequency/frequency_model_analysis.html")
fig.write_image("figures/frequency/frequency_model_analysis.png", width=fig_width, height=700, scale=2)

In [ ]:
# Plot bias distributions to check if they are being used
px.histogram(model.left.bias).show()
px.histogram(model.right.bias).show()

# Weight interpretation

In [13]:
def get_marker_gene_lists(
    gene_names: np.ndarray,
    vecs: np.ndarray,
    n_modules: int = 1,
    n_top_genes: int = 50,
) -> np.ndarray:
    """Extract marker genes optimized for GO analysis."""
    gene_lists = []
    for i in range(n_modules):
        top_idxs = vecs[:,i].topk(n_top_genes).indices
        top_genes = gene_names[top_idxs].tolist()
        bottom_idxs = (-vecs[:,i]).topk(n_top_genes).indices
        bottom_genes = gene_names[bottom_idxs].tolist()
        gene_lists.append([top_genes, bottom_genes])
    return np.array(gene_lists)

def decompose_gene_weights(model, out_idx):
    """Perform eigendecomposition of bilinear weights for a specific output."""
    q = einops.einsum(model.w_p[out_idx], model.w_l, model.w_r, "hid, hid in1, hid in2 -> in1 in2")
    q = 0.5 * (q + q.T)  # symmetrize
    vals, vecs = torch.linalg.eigh(q)
    vals = vals.flip([0])
    vecs = vecs.flip([1])
    return vals, vecs


def print_gene_modules(gene_name, gene_lists, n_modules=3):
    """Print gene modules for a specific gene."""
    print("="*20, gene_name, "="*20)
    for module_idx in range(n_modules):
        print("="*20, "Module", module_idx, "="*20)
        print(f"Positive genes: {gene_lists[module_idx,0,:10]}...")
        print(f"Negative genes: {gene_lists[module_idx,1,:10]}...")


def analyze_go_terms(gene_lists, n_modules=3, n_results=5):
    """Perform GO term analysis on gene modules."""
    print("\n" + "="*50)
    print("GO ANALYSIS")
    print("="*50)

    results_cols = ["Term", "Genes", "Gene_set", "Adjusted P-value"]

    for module_idx in range(n_modules):
        print("="*10, "Module", module_idx, "="*10)

        # Use Enrichr with optimized gene sets for pancreatic development
        for i in range(2):
            side = "Positive" if i == 0 else "Negative"
            print(f"\n--- {side} genes ---")
            enr = gp.enrichr(
                gene_list=gene_lists[module_idx, i].tolist(),
                gene_sets=[
                    "GO_Biological_Process_2023",
                    "GO_Molecular_Function_2023",
                    "GO_Cellular_Component_2023", 
                    "KEGG_2019_Mouse",
                    "Reactome_2022",
                    "MSigDB_Hallmark_2020",
                    "WikiPathways_2019_Mouse"
                ],
                organism="mouse",
            )
            # Filter and combine results from all gene sets
            all_results = enr.results[enr.results['Adjusted P-value'] <= 0.05].copy()
            if len(all_results) > 0:
                # Sort by p-value and show top results
                all_results = all_results.sort_values('Adjusted P-value')
                display(all_results.head(n_results)[results_cols])
            else:
                print(f"No significant results found (p < 0.05)")

In [14]:
# Shared parameters for frequency module analysis
n_modules = 3
n_top_genes = int(0.01*n_genes)  # top 1% of genes
gene_names = adata.var_names.values
n_results = 15

## Frequency 0

In [ ]:
freq_idx = 0

In [ ]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, freq_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f})", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Frequency 0 (λ=0.0272) ====================
==================== Module 0 ====================
Positive genes: ['Malat1' 'Gnas' 'Pcsk1n' 'Cpe' 'Pfn1' 'Rbp4' 'Iapp' 'Pyy' 'Dynll1'
 'Aplp1']...
Negative genes: ['Gas6' 'Cd24a' 'Sox9' 'Csrp1' 'Col9a3' 'Bicc1' 'Rbp2' 'Lamb1' 'Maob'
 'Pi4k2b']...
==================== Module 1 ====================
Positive genes: ['Tmsb10' 'Marcksl1' 'Dynll1' '2810417H13Rik' 'Myl6' 'Jun' 'Malat1'
 'Mpzl1' 'Birc5' 'Pnliprp1']...
Negative genes: ['Eef1a1' 'Ins2' 'Ins1' 'Iapp' 'Rbp4' 'Dlk1' 'Sec61b' 'Chga' 'Pfn1'
 'Hmgn3']...
==================== Module 2 ====================
Positive genes: ['Ghrl' 'Tmsb4x' 'Gcg' 'Eef1a1' 'Rbp4' 'Hsp90aa1' 'Isl1' 'Ssr2' 'Ttr'
 'Cck']...
Negative genes: ['Ins2' 'Ins1' 'Malat1' 'Spp1' 'Tmsb10' 'Nnat' 'Calm1' 'Pdia6' 'Tsc22d1'
 'Sh3bgrl3']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1612,Pancreas Beta Cells,CHGA;MAFB;SST;PAX6;GCG;IAPP;SYT13;ISL1,MSigDB_Hallmark_2020,7.520697e-10
824,Hormone Activity (GO:0005179),PYY;SST;NPY;PPY;GCG;GHRL;CHGB,GO_Molecular_Function_2023,1.235559e-05
0,Regulation Of Insulin Secretion (GO:0050796),CHGA;GLUD1;UQCC2;GCG;GHRL;ISL1;SOX4,GO_Biological_Process_2023,2.192627e-04
1,Positive Regulation Of Insulin Secretion (GO:0...,GLUD1;GCG;GHRL;ISL1;SOX4,GO_Biological_Process_2023,6.665516e-04
1217,Peptide Hormone Metabolism R-HSA-2980736,CHGA;CPE;PAX6;GCG;GHRL;ISL1,Reactome_2022,2.204439e-03
1218,"Tetrahydrobiopterin (BH4) Synthesis, Recycling...",HSP90AA1;GCH1;CALM1,Reactome_2022,2.802112e-03
2,Columnar/Cuboidal Epithelial Cell Differentiat...,PYY;NPY;SOX4,GO_Biological_Process_2023,3.896945e-03
1220,Response To Elevated Platelet Cytosolic Ca2+ R...,CD63;TTR;TMSB4X;TAGLN2;PFN1;CALM1,Reactome_2022,4.796442e-03
1219,Platelet Degranulation R-HSA-114608,CD63;TTR;TMSB4X;TAGLN2;PFN1;CALM1,Reactome_2022,4.796442e-03
825,Neuropeptide Receptor Binding (GO:0071855),PYY;NPY;PPY,GO_Molecular_Function_2023,5.604048e-03



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
0,Urogenital System Development (GO:0001655),LHX1;SOX9;EPHB3,GO_Biological_Process_2023,0.012429
1040,Endoplasmic Reticulum Lumen (GO:0005788),SPP1;SERPINH1;COL9A3;LAMB1;COL9A2;GAS6;F3,GO_Cellular_Component_2023,0.026995
1041,Collagen-Containing Extracellular Matrix (GO:0...,MFAP4;SPARC;ANXA2;SERPINH1;COL9A3;LAMB1;COL9A2...,GO_Cellular_Component_2023,0.026995


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1877,Pancreas Beta Cells,PCSK2;CHGA;ABCC8;PDX1;IAPP;NKX6-1;PAK3;ISL1,MSigDB_Hallmark_2020,7.322784e-10
1240,Protein processing in endoplasmic reticulum,PDIA3;DNAJA1;HSP90AA1;HSPA5;RPN2;SSR2;RPN1;CAL...,KEGG_2019_Mouse,8.587294e-08
1878,mTORC1 Signaling,FKBP2;SDF2L1;SERP1;HSPA5;PSMC2;RPN1;SERPINH1;C...,MSigDB_Hallmark_2020,9.154933e-08
1241,Thyroid hormone synthesis,GPX2;GPX1;TTR;HSPA5;GNAS;ATP1B1;PDIA4,KEGG_2019_Mouse,5.930418e-06
1243,Insulin secretion,INS1;RAB3A;ABCC8;INS2;PDX1;GNAS;ATP1B1,KEGG_2019_Mouse,9.274548e-06
1242,Maturity onset diabetes of the young,INS1;INS2;PDX1;IAPP;NKX6-1,KEGG_2019_Mouse,9.274548e-06
1113,Intracellular Organelle Lumen (GO:0070013),PDIA3;APP;HSP90AA1;GPX1;HSPA5;IDH2;PYCR2;PDIA6...,GO_Cellular_Component_2023,1.386945e-04
1392,Cellular Responses To Stimuli R-HSA-8953897,GPX2;HSP90AA1;GPX1;HSPA5;PDIA6;LRPPRC;EEF1A1;D...,Reactome_2022,1.760398e-04
1391,Cellular Responses To Stress R-HSA-2262752,GPX2;HSP90AA1;GPX1;HSPA5;PDIA6;LRPPRC;EEF1A1;D...,Reactome_2022,1.760398e-04
1879,Unfolded Protein Response,SERP1;HSPA5;DNAJB9;CALR;ATP6V0D1;PDIA6,MSigDB_Hallmark_2020,2.717384e-04


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
821,Polymeric Cytoskeletal Fiber (GO:0099513),BEX4;TUBA1B;KRT18;TUBA1A;KIF2A;KRT8;TWF2;KRT7;...,GO_Cellular_Component_2023,0.000092
1048,Peptide Hormone Metabolism R-HSA-2980736,ANPEP;CTSZ;GNB1;GCG;GHRL;MBOAT4;ISL1,Reactome_2022,0.000189
822,Secretory Granule Lumen (GO:0034774),EEF1A1;HSP90AA1;TTR;ARG1;TMSB4X;PSMC2;CTSZ;GCG...,GO_Cellular_Component_2023,0.000224
687,Hormone Activity (GO:0005179),PYY;GAST;CCK;GCG;GHRL,GO_Molecular_Function_2023,0.004640
823,Cytoplasmic Vesicle Lumen (GO:0060205),EEF1A1;HSP90AA1;PSMC2;GCG;GHRL,GO_Cellular_Component_2023,0.009049
824,Microtubule (GO:0005874),BEX4;TUBA1B;TUBA1A;KIF2A;TUBA4A;LRPPRC,GO_Cellular_Component_2023,0.009049
1057,Formation Of Tubulin Folding Intermediates By ...,TUBA1B;TUBA1A;TUBA4A,Reactome_2022,0.017606
1055,Drug-mediated Inhibition Of CDK4/CDK6 Activity...,CCND2;CDK4,Reactome_2022,0.017606
1054,"Incretin Synthesis, Secretion, And Inactivatio...",GNB1;GCG;ISL1,Reactome_2022,0.017606
1053,Post-chaperonin Tubulin Folding Pathway R-HSA-...,TUBA1B;TUBA1A;TUBA4A,Reactome_2022,0.017606



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1876,Pancreas Beta Cells,G6PC2;PDX1;SYT13;ELP4;INSM1;NKX6-1,MSigDB_Hallmark_2020,0.000002
1269,Maturity onset diabetes of the young,INS1;INS2;PDX1;HES1;NKX6-1,KEGG_2019_Mouse,0.000040
1877,mTORC1 Signaling,MLLT11;BTG2;SDF2L1;TPI1;IDH1;ENO1;CALR;GAPDH,MSigDB_Hallmark_2020,0.000151
0,Regulation Of Insulin Secretion (GO:0050796),G6PC2;UQCC2;NNAT;PDX1;SOX4;ADRA2A;GIP,GO_Biological_Process_2023,0.000259
1878,Hypoxia,CDKN1C;JUN;TPI1;CSRP2;MIF;ENO1;GAPDH,MSigDB_Hallmark_2020,0.000898
1917,Ptf1a related regulatory pathway WP201,PDX1;HES1;NKX6-1,WikiPathways_2019_Mouse,0.001438
1918,Adipogenesis genes WP447,INS1;FRZB;GADD45A;INS2;MIF;DLK1,WikiPathways_2019_Mouse,0.002129
1,Regulation Of Cell Population Proliferation (G...,JUN;BTG2;TSC22D1;CD81;HMGB2;DYNLL1;ADRA2A;CLDN...,GO_Biological_Process_2023,0.002914
1879,E2F Targets,PRPS1;HMGB2;HMGB3;GSPT1;SPC24;CKS1B,MSigDB_Hallmark_2020,0.003465
1880,Glycolysis,PRPS1;CLDN3;TPI1;IDH1;MIF;ENO1,MSigDB_Hallmark_2020,0.003465


In [ ]:
freq = adata.obsm["X_freq"][:, freq_idx]
gene_latent = adata.layers[layer] @ vecs[:,:3]

# Create horizontal subplots
subplot_titles = [f"Module {i}" for i in range(3)]
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=subplot_titles,
    horizontal_spacing=0.08,
    shared_yaxes=False
)

# Add traces for each module
for module_idx in range(3):
    for i, cluster in enumerate(unique_clusters):
        mask = adata.obs["clusters"] == cluster
        fig.add_trace(
            go.Scatter(
                x=freq[mask],
                y=gene_latent[mask, module_idx],
                mode="markers",
                marker=dict(
                    size=5,
                    color=cluster_colors[cluster]
                ),
                name=cluster,
                showlegend=(module_idx == 0),  # Only show legend for first subplot
                legendgroup=cluster,  # Group by cluster for consistent legend
                hovertemplate=f"Frequency: %{{x}}<br>Module {module_idx}: %{{y}}<br>Cluster: {cluster}<extra></extra>"
            ),
            row=1, col=module_idx + 1
        )

# Update layout for square subplots with shared legend
fig.update_layout(
    title_text=f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f}) vs Gene Modules",
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    width=1200,  # Wider to accommodate 3 square subplots
    height=400,   # Height for square subplots
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.02,
        font=dict(size=legend_fontsize-2)
    ),
    margin=dict(l=0, r=0, t=80, b=60)  # Extra right margin for legend
)

# Update axes labels
fig.update_xaxes(title_text="Frequency", title_font=dict(size=legend_fontsize-2))
fig.update_yaxes(title_text="Module Value", title_font=dict(size=legend_fontsize-2), row=1, col=1)  # Only leftmost plot

fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_modules.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_modules.png", width=fig_width+150, height=370, scale=2)

In [ ]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_eigvals.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_eigvals.png", width=fig_width, height=300, scale=2)

## Frequency 1

In [72]:
freq_idx = 1

In [73]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, freq_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f})", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Frequency 1 (λ=0.1627) ====================
==================== Module 0 ====================
Positive genes: ['Hspa5' 'Tuba1a' 'Epcam' 'Jun' 'Ssr2' 'Cd81' 'Tubb5' 'Cdk4' 'Ier2'
 'Pdia6']...
Negative genes: ['Eef1a1' 'Arf5' 'Hsp90aa1' 'Ngfrap1' 'Gabarapl2' 'Eif4a2' 'Clps'
 'Slc25a5' 'Gcg' 'Krt18']...
==================== Module 1 ====================
Positive genes: ['Cdk4' 'Ier2' 'Tuba1b' 'Tuba1a' 'Bex2' 'Mt1' 'Trappc2l' 'Hadh' 'Hspa5'
 'Ambp']...
Negative genes: ['Arf5' 'Gapdh' 'Hmgn3' 'Ctsb' 'Krt18' 'Hmgb1' 'Npc2' 'Eef1a1' 'Nt5dc2'
 'App']...
==================== Module 2 ====================
Positive genes: ['Calm1' 'Cpe' 'Pcsk1n' 'Jun' 'Hmgn3' 'Pfn1' 'Gars' 'Uqcc2' 'Gch1' 'Chgb']...
Negative genes: ['Tmsb4x' 'Clps' 'Spint2' 'Tmsb10' 'H2afv' 'Cdk4' 'Aplp1' 'Hsp90aa1'
 'Jund' 'Peg3']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1149,Protein processing in endoplasmic reticulum,HSPA8;HSPA5;RPN2;SSR2;SSR1;CALR;PDIA6;HSP90B1;...,KEGG_2019_Mouse,0.000019
1745,Unfolded Protein Response,HSPA5;SSR1;CALR;ATP6V0D1;PDIA6;HSP90B1;CKS1B,MSigDB_Hallmark_2020,0.000031
1746,Pancreas Beta Cells,SCGN;MAFB;PAX4;IAPP;NEUROG3,MSigDB_Hallmark_2020,0.000031
1150,Phagosome,RAB7;TUBA1B;TUBB5;TUBA1A;TUBB3;CALR;ATP6V1E1;A...,KEGG_2019_Mouse,0.000249
1296,RHO GTPases Activate ROCKs R-HSA-5627117,MYL6;ROCK1;RHOC;MYL12B,Reactome_2022,0.000602
1297,Sema4D Induced Cell Migration And Growth-Cone ...,MYL6;ROCK1;RHOC;MYL12B,Reactome_2022,0.000602
1298,Unfolded Protein Response (UPR) R-HSA-381119,HSPA5;SSR1;CALR;ATP6V0D1;PDIA6;HSP90B1,Reactome_2022,0.000650
1299,Sema4D In Semaphorin Signaling R-HSA-400685,MYL6;ROCK1;RHOC;MYL12B,Reactome_2022,0.000650
1300,ATF6 (ATF6-alpha) Activates Chaperone Genes R-...,HSPA5;CALR;HSP90B1,Reactome_2022,0.000895
1151,Tight junction,MYL6;JUN;TUBA1B;TUBA1A;ROCK1;CDK4;MYL12B,KEGG_2019_Mouse,0.001020



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
741,Endopeptidase Activity (GO:0004175),PCSK2;AFG3L2;ADAMTS5;PRSS50;ADAMTS1;RHBDD1;ERA...,GO_Molecular_Function_2023,0.007074
742,Ubiquitin Protein Ligase Binding (GO:0031625),GABARAPL2;GSK3B;HSP90AA1;SUMO2;TXNIP;SLC25A5;L...,GO_Molecular_Function_2023,0.023669
743,Hormone Activity (GO:0005179),NPY;GCG;GHRL;CHGB,GO_Molecular_Function_2023,0.023669
744,Ubiquitin-Like Protein Ligase Binding (GO:0044...,GABARAPL2;GSK3B;HSP90AA1;SUMO2;TXNIP;SLC25A5;L...,GO_Molecular_Function_2023,0.023669


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
1067,Secretory Granule Lumen (GO:0034774),EEF1A1;APP;HSP90AA1;CYB5R3;NPC2;FUCA1;GHRL;HMG...,GO_Cellular_Component_2023,0.000327
1068,Intracellular Organelle Lumen (GO:0070013),PDIA3;APP;CD63;HSP90AA1;UQCC2;TXNDC12;HMGB1;HS...,GO_Cellular_Component_2023,0.000327
1069,Endosome Lumen (GO:0031904),APP;CD63;CTSL;CTSB,GO_Cellular_Component_2023,0.000327
1070,Lysosomal Lumen (GO:0043202),HSP90AA1;CTSL;NPC2;FUCA1;CTSB,GO_Cellular_Component_2023,0.002553
1071,Vacuolar Lumen (GO:0005775),HSP90AA1;CYB5R3;NPC2;CTSL;FUCA1;TUBB4B,GO_Cellular_Component_2023,0.004528
1349,"Tetrahydrobiopterin (BH4) Synthesis, Recycling...",HSP90AA1;GCH1;CALM1,Reactome_2022,0.005091
1350,Neutrophil Degranulation R-HSA-6798695,EEF1A1;CD63;HSP90AA1;CYB5R3;NPC2;FUCA1;CYSTM1;...,Reactome_2022,0.005091
1210,Lysosome,CD63;ATP6V0B;NPC2;CTSL;FUCA1;CTSB,KEGG_2019_Mouse,0.005179
1806,Apoptosis,APP;KRT18;GCH1;LMNA;TXNIP;SQSTM1,MSigDB_Hallmark_2020,0.005858
1211,Antigen processing and presentation,PDIA3;HSP90AA1;CTSL;CANX;CTSB,KEGG_2019_Mouse,0.006164


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1198,Protein processing in endoplasmic reticulum,HSPA8;HSPA5;TUSC3;SSR2;RPN1;SEC61B;PDIA6;PDIA4...,KEGG_2019_Mouse,0.000022
1070,Vesicle (GO:0031982),TUBB2A;CD81;PEG10;ANXA4;STX7;ATP1B1;CALM1;TUBB...,GO_Cellular_Component_2023,0.000036
1199,Phagosome,ATP6V0B;TUBB2A;TUBA1A;CTSL;STX7;SEC61B;ATP6V1E...,KEGG_2019_Mouse,0.000286
1860,mTORC1 Signaling,HSPA5;PSMC2;RPN1;PHGDH;HSPE1;ALDOA;GAPDH,MSigDB_Hallmark_2020,0.000876
1859,Complement,HSPA5;CTSL;SCG3;PFN1;CALM1;F3;ATOX1,MSigDB_Hallmark_2020,0.000876
1858,TNF-alpha Signaling via NF-kB,EGR1;JUN;TUBB2A;TSC22D1;GCH1;ID2;F3,MSigDB_Hallmark_2020,0.000876
1861,Protein Secretion,KRT18;TMX1;STX7;GNAS;PAM,MSigDB_Hallmark_2020,0.001205
1071,Secretory Granule Lumen (GO:0034774),APP;HSPA8;TUBB2A;NPC2;PSMC2;GHRL;ALDOA;TUBB4B;F3,GO_Cellular_Component_2023,0.001891
1863,KRAS Signaling Up,PCSK1N;MAFB;ID2;TMEM176B;CPE;SCG3,MSigDB_Hallmark_2020,0.003380
1862,Hypoxia,JUN;HSPA5;ALDOA;PAM;F3;GAPDH,MSigDB_Hallmark_2020,0.003380



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
2060,E2F Targets,TOP2A;RANBP1;CDKN1A;RRM2;RACGAP1;CDK4;HMGB2;BI...,MSigDB_Hallmark_2020,1.397848e-08
2061,G2-M Checkpoint,TOP2A;MARCKS;RACGAP1;CDK4;BIRC5;TOP1;BUB3;NDC8...,MSigDB_Hallmark_2020,9.897225e-08
1394,"Cell Cycle, Mitotic R-HSA-69278",TOP2A;HSP90AA1;FEN1;CDKN1A;RRM2;NDC80;CKS1B;CD...,Reactome_2022,1.368916e-03
0,DNA Topological Change (GO:0006265),TOP2A;HMGB2;TOP1,GO_Biological_Process_2023,6.309208e-03
2062,Epithelial Mesenchymal Transition,SPARC;TPM4;APLP1;TPM1;SPP1;PPIB,MSigDB_Hallmark_2020,6.760398e-03
1395,Cell Cycle R-HSA-1640170,TOP2A;HSP90AA1;FEN1;CDKN1A;RRM2;NDC80;CKS1B;CD...,Reactome_2022,7.536442e-03
2100,Signal Transduction of S1P Receptor WP57,RACGAP1;GNAI2;MAPK3,WikiPathways_2019_Mouse,8.500681e-03
2101,Novel Jun-Dmp1 Pathway WP3654,CDK4;TRP53;MAPK3,WikiPathways_2019_Mouse,8.500681e-03
2102,p53 signaling WP2902,CDKN1A;RRM2;CDK4;TRP53,WikiPathways_2019_Mouse,8.500681e-03
1225,Bladder cancer,CDKN1A;CDK4;TRP53;MAPK3,KEGG_2019_Mouse,8.735526e-03


In [ ]:
freq = adata.obsm["X_freq"][:, freq_idx]
gene_latent = adata.layers[layer] @ vecs[:,:3]

# Create horizontal subplots
subplot_titles = [f"Module {i}" for i in range(3)]
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=subplot_titles,
    horizontal_spacing=0.08,
    shared_yaxes=False
)

# Add traces for each module
for module_idx in range(3):
    for i, cluster in enumerate(unique_clusters):
        mask = adata.obs["clusters"] == cluster
        fig.add_trace(
            go.Scatter(
                x=freq[mask],
                y=gene_latent[mask, module_idx],
                mode="markers",
                marker=dict(
                    size=5,
                    color=cluster_colors[cluster]
                ),
                name=cluster,
                showlegend=(module_idx == 0),  # Only show legend for first subplot
                legendgroup=cluster,  # Group by cluster for consistent legend
                hovertemplate=f"Frequency: %{{x}}<br>Module {module_idx}: %{{y}}<br>Cluster: {cluster}<extra></extra>"
            ),
            row=1, col=module_idx + 1
        )

# Update layout for square subplots with shared legend
fig.update_layout(
    title_text=f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f}) vs Gene Modules",
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    width=1200,  # Wider to accommodate 3 square subplots
    height=400,   # Height for square subplots
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.02,
        font=dict(size=legend_fontsize-2)
    ),
    margin=dict(l=0, r=0, t=80, b=60)  # Extra right margin for legend
)

# Update axes labels
fig.update_xaxes(title_text="Frequency", title_font=dict(size=legend_fontsize-2))
fig.update_yaxes(title_text="Module Value", title_font=dict(size=legend_fontsize-2), row=1, col=1)  # Only leftmost plot

fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_modules.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_modules.png", width=fig_width+150, height=370, scale=2)

In [ ]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_eigvals.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_eigvals.png", width=fig_width, height=300, scale=2)

## Frequency 2

In [76]:
freq_idx = 2

In [77]:
# Decompose weights and extract gene modules
vals, vecs = decompose_gene_weights(model, freq_idx)
gene_lists = get_marker_gene_lists(
    gene_names, vecs, n_modules=n_modules, n_top_genes=n_top_genes
)

# Print gene modules
print_gene_modules(f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f})", gene_lists, n_modules=n_modules)

# GO term analysis
analyze_go_terms(gene_lists, n_modules=n_modules, n_results=n_results)

==================== Frequency 2 (λ=0.4256) ====================
==================== Module 0 ====================
Positive genes: ['Ghrl' 'Gcg' 'Rbp4' 'Isl1' 'Pfn1' 'Ssr2' 'Malat1' 'Iapp' 'Ngfrap1' 'Ttr']...
Negative genes: ['Ins2' 'Ins1' 'Npy' 'Adra2a' 'Nnat' 'Gip' 'Sytl4' 'Spock2' 'Scaper'
 'Mapt']...
==================== Module 1 ====================
Positive genes: ['Tmsb4x' 'Clps' 'Rbp4' 'Ghrl' 'Aplp1' 'Ttr' 'Gcg' 'Hsp90aa1' 'Eef1a1'
 'Cck']...
Negative genes: ['Calm1' 'Spp1' 'Nnat' 'Malat1' 'Jun' 'Tsc22d1' 'Gars' 'Pdia6' 'Pfn1'
 'Gapdh']...
==================== Module 2 ====================
Positive genes: ['Spp1' 'Myl12a' 'Rbp4' 'Dynll1' 'Cpa1' 'Hsp90aa1' 'Hmgb2' 'Tmsb10' 'Clu'
 'Mif']...
Negative genes: ['Pyy' 'Cpe' 'Ypel3' 'Neurog3' 'Ssr2' 'Hmgn3' 'Tuba1a' 'Psmc2' 'Jun'
 'Xist']...

GO ANALYSIS
========== Module 0 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
1198,Protein processing in endoplasmic reticulum,PDIA3;HSPA8;HSP90AA1;HSPA5;SSR4;TUSC3;SSR2;CAN...,KEGG_2019_Mouse,4.357923e-09
1333,Peptide Hormone Metabolism R-HSA-2980736,ANPEP;CTSZ;FFAR4;CPE;GCG;GHRL;MBOAT4;ISL1,Reactome_2022,5.933667e-06
1067,Secretory Granule Lumen (GO:0034774),EEF1A1;HSPA8;HSP90AA1;TTR;NPC2;ARG1;CTSZ;GCG;M...,GO_Cellular_Component_2023,7.811612e-06
1199,Phagosome,ATP6V0B;TUBA1B;TUBB5;TUBA1A;CANX;CALR;SEC61B;T...,KEGG_2019_Mouse,2.015098e-05
1334,Metabolism Of Proteins R-HSA-392499,TUSC3;CTSZ;HSP90B1;TUBA1B;TUBA1A;TTR;ANPEP;FFA...,Reactome_2022,7.399860e-05
1732,mTORC1 Signaling,HSPA5;CANX;CALR;HSPE1;ALDOA;GAPDH;TUBA4A;HSP90B1,MSigDB_Hallmark_2020,2.651765e-04
1200,Antigen processing and presentation,PDIA3;HSPA8;HSP90AA1;HSPA5;CANX;CALR,KEGG_2019_Mouse,2.679969e-04
1335,Neutrophil Degranulation R-HSA-6798695,EEF1A1;HSPA8;PRDX5;HSP90AA1;PGRMC1;TTR;NPC2;AN...,Reactome_2022,4.830522e-04
1336,Post-chaperonin Tubulin Folding Pathway R-HSA-...,TUBA1B;TUBA1A;TUBB4B;TUBA4A,Reactome_2022,4.830522e-04
906,GTP Binding (GO:0005525),EEF1A1;GLUD1;TUBA1B;TUBA1A;GCH1;TUBB4B;TUBA4A;RAN,GO_Molecular_Function_2023,6.188422e-04



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
945,Maturity onset diabetes of the young,INS1;RFX6;INS2;PDX1,KEGG_2019_Mouse,0.000983
0,Regulation Of Insulin Secretion (GO:0050796),G6PC2;RFX6;NNAT;PDX1;ADRA2A;GIP,GO_Biological_Process_2023,0.003427
1,Regulation Of Organelle Organization (GO:0033043),CAMSAP3;MIEF2;MAPT;EZR;CAMSAP2,GO_Biological_Process_2023,0.037465
2,Regulation Of Protein Secretion (GO:0050708),G6PC2;RFX6;EZR;SYTL4;GIP,GO_Biological_Process_2023,0.037465
3,Regulation Of Supramolecular Fiber Organizatio...,CAMSAP3;MAPT;CAMSAP2,GO_Biological_Process_2023,0.039858
4,Regulation Of Microtubule Polymerization Or De...,CAMSAP3;MAPT;CAMSAP2,GO_Biological_Process_2023,0.045582
5,Synapse Organization (GO:0050808),BDNF;SPOCK2;RAB39B;GPC4;MAPT,GO_Biological_Process_2023,0.046220
6,ADP Transport (GO:0015866),SLC25A25;SLC25A24,GO_Biological_Process_2023,0.046220
7,Positive Regulation Of Protein Secretion (GO:0...,NNAT;PDX1;EZR;SYTL4,GO_Biological_Process_2023,0.046220
8,Regulation Of Protein Polymerization (GO:0032271),CAMSAP3;MAPT;CAMSAP2,GO_Biological_Process_2023,0.046220


========== Module 1 ==========

--- Positive genes ---


...

,Term,Genes,Gene_set,Adjusted P-value
2027,mTORC1 Signaling,BTG2;SDF2L1;HSPA5;PSMC2;SERPINH1;PHGDH;ENO1;CA...,MSigDB_Hallmark_2020,0.000003
1304,Maturity onset diabetes of the young,INS1;INS2;PDX1;IAPP;NKX6-1,KEGG_2019_Mouse,0.000038
2028,Pancreas Beta Cells,CHGA;PDX1;IAPP;NKX6-1,MSigDB_Hallmark_2020,0.000690
2029,Oxidative Phosphorylation,CYB5A;CYB5R3;MAOB;ECH1;IDH2;ATP6V1E1;ATP1B1,MSigDB_Hallmark_2020,0.000690
2030,Glycolysis,CYB5A;CLDN3;NASP;HSPA5;ENO1;SOX9;CENPA,MSigDB_Hallmark_2020,0.000690
2069,Endochondral Ossification WP1270,CDKN1C;SPP1;SERPINH1;SOX9;CALM1,WikiPathways_2019_Mouse,0.001065
2031,G2-M Checkpoint,HSPA8;NASP;NCL;HMGB3;CENPA;CKS1B,MSigDB_Hallmark_2020,0.004259
2032,Apoptosis,APP;JUN;BTG2;GPX1;HMGB2,MSigDB_Hallmark_2020,0.009049
0,Chaperone Cofactor-Dependent Protein Refolding...,HSPA8;SDF2L1;HSPA5;HSPE1,GO_Biological_Process_2023,0.009289
2,Regulation Of Cardiac Muscle Contraction (GO:0...,CHGA;SRI;ATP1B1;CALM1,GO_Biological_Process_2023,0.009289


========== Module 2 ==========

--- Positive genes ---


,Term,Genes,Gene_set,Adjusted P-value
976,Ubiquitin-Like Protein Ligase Binding (GO:0044...,DNAJA1;GABARAPL2;HSPA8;CCNB1;HSP90AA1;TUBA1B;T...,GO_Molecular_Function_2023,0.000003
977,Ubiquitin Protein Ligase Binding (GO:0031625),DNAJA1;GABARAPL2;HSPA8;HSP90AA1;TUBA1B;TPI1;TU...,GO_Molecular_Function_2023,0.000008
1949,E2F Targets,CDC20;CCNB2;CDCA3;SRSF2;HMGB2;BIRC5;CDCA8;PAIC...,MSigDB_Hallmark_2020,0.000030
978,Supercoiled DNA Binding (GO:0097100),HMGB2;HMGB1;TOP1,GO_Molecular_Function_2023,0.000058
1950,G2-M Checkpoint,CDC20;CCNB2;HSPA8;SRSF2;BIRC5;TOP1;CENPA;CKS1B,MSigDB_Hallmark_2020,0.000151
1437,Resolution Of Sister Chromatid Cohesion R-HSA-...,CDC20;CCNB2;CCNB1;BIRC5;CDCA8;DYNLL1;CENPA,Reactome_2022,0.000403
1438,Mitotic Anaphase R-HSA-68882,CDC20;CCNB2;CCNB1;TUBA1B;TUBB2A;BIRC5;CDCA8;DY...,Reactome_2022,0.000403
1439,Mitotic Metaphase And Anaphase R-HSA-2555396,CDC20;CCNB2;CCNB1;TUBA1B;TUBB2A;BIRC5;CDCA8;DY...,Reactome_2022,0.000403
1440,M Phase R-HSA-68886,CDC20;CCNB2;CCNB1;HSP90AA1;TUBA1B;TUBB2A;BIRC5...,Reactome_2022,0.000403
1441,Mitotic Prometaphase R-HSA-68877,CDC20;CCNB2;CCNB1;HSP90AA1;BIRC5;CDCA8;DYNLL1;...,Reactome_2022,0.000442



--- Negative genes ---


,Term,Genes,Gene_set,Adjusted P-value
1905,Pancreas Beta Cells,CHGA;MAFB;GCG;SYT13;NEUROG3,MSigDB_Hallmark_2020,0.000065
1906,Apoptosis,APP;JUN;GCH1;GADD45A;GPX3;TXNIP,MSigDB_Hallmark_2020,0.003167
1223,Phagosome,ATP6V0B;TUBB5;TUBA1A;TUBB3;CTSL;ATP6V1E1;TUBA4A,KEGG_2019_Mouse,0.004327
1224,Gap junction,TUBB5;TUBA1A;TUBB3;GNAS;TUBA4A,KEGG_2019_Mouse,0.004327
1225,Apoptosis,JUN;TUBA1A;GADD45A;CTSL;TUBA4A;CTSB,KEGG_2019_Mouse,0.004327
1393,Platelet Degranulation R-HSA-114608,APP;TTR;TMSB4X;PFN1;CALM1;TUBA4A,Reactome_2022,0.008984
1394,Response To Elevated Platelet Cytosolic Ca2+ R...,APP;TTR;TMSB4X;PFN1;CALM1;TUBA4A,Reactome_2022,0.008984
1395,Innate Immune System R-HSA-168249,CHGA;APP;ATP6V0B;JUN;FUCA1;SURF4;EEF1A1;PPP3CA...,Reactome_2022,0.008984
1226,Amphetamine addiction,PPP3CA;JUN;GNAS;CALM1,KEGG_2019_Mouse,0.013514
1227,Lysosome,ATP6V0B;NPC2;CTSL;FUCA1;CTSB,KEGG_2019_Mouse,0.013514


In [ ]:
freq = adata.obsm["X_freq"][:, freq_idx]
gene_latent = adata.layers[layer] @ vecs[:,:3]

# Create horizontal subplots
subplot_titles = [f"Module {i}" for i in range(3)]
fig = make_subplots(
    rows=1, cols=3, 
    subplot_titles=subplot_titles,
    horizontal_spacing=0.08,
    shared_yaxes=False
)

# Add traces for each module
for module_idx in range(3):
    for i, cluster in enumerate(unique_clusters):
        mask = adata.obs["clusters"] == cluster
        fig.add_trace(
            go.Scatter(
                x=freq[mask],
                y=gene_latent[mask, module_idx],
                mode="markers",
                marker=dict(
                    size=5,
                    color=cluster_colors[cluster]
                ),
                name=cluster,
                showlegend=(module_idx == 0),  # Only show legend for first subplot
                legendgroup=cluster,  # Group by cluster for consistent legend
                hovertemplate=f"Frequency: %{{x}}<br>Module {module_idx}: %{{y}}<br>Cluster: {cluster}<extra></extra>"
            ),
            row=1, col=module_idx + 1
        )

# Update layout for square subplots with shared legend
fig.update_layout(
    title_text=f"Frequency {freq_idx} (λ={adata.uns['freq_vals'][freq_idx]:.4f}) vs Gene Modules",
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),
    width=1200,  # Wider to accommodate 3 square subplots
    height=400,   # Height for square subplots
    legend=dict(
        orientation="v",
        yanchor="middle",
        y=0.5,
        xanchor="left",
        x=1.02,
        font=dict(size=legend_fontsize-2)
    ),
    margin=dict(l=0, r=0, t=80, b=60)  # Extra right margin for legend
)

# Update axes labels
fig.update_xaxes(title_text="Frequency", title_font=dict(size=legend_fontsize-2))
fig.update_yaxes(title_text="Module Value", title_font=dict(size=legend_fontsize-2), row=1, col=1)  # Only leftmost plot

fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_modules.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_modules.png", width=fig_width+150, height=370, scale=2)

In [ ]:
n_vals = 5
fig = px.bar(
    x=list(range(n_vals)),
    y=vals.numpy()[:n_vals],
    labels={"x": "Module", "y": "Eigenvalue"},
    title="Top Module Eigenvalues"
)
fig.update_layout(
    title_font=dict(size=title_fontsize),
    font=dict(size=legend_fontsize),  # base font (ticks/legend unless overridden)
    legend=dict(font=dict(size=legend_fontsize)),
    margin=dict(l=200, r=200, t=100, b=50)
)
fig.update_xaxes(title_font=dict(size=legend_fontsize))
fig.update_yaxes(title_font=dict(size=legend_fontsize))
fig.show()

# Save plot
# fig.write_html(f"figures/frequency/frequency_{freq_idx}_eigvals.html")
fig.write_image(f"figures/frequency/frequency_{freq_idx}_eigvals.png", width=fig_width, height=300, scale=2)